# Can we predict next-month churn?
Target: `churn_next_month`. One row per account per monthly snapshot (~14.8k rows, 3,000 accounts).

Plan: check data quality → audit features for leakage → build a validation scheme that mimics real use (train on the past, predict the future) → compare models against baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv'); df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
print(df.shape, '| accounts:', df.account_ref.nunique(), '| dup account-months:', df.duplicated(['account_ref','snapshot_month']).sum(), '| nulls:', df.isna().sum().sum())
print('churn rate: %.3f' % df.churn_next_month.mean())
df.describe().T

## 1. Churn rate over time
Churn is strongly seasonal / non-stationary (a spike around Nov–Jan snapshots), which matters for how we validate.

In [ ]:
m = df.groupby('month').churn_next_month.agg(['mean','size'])
fig, ax = plt.subplots(figsize=(9,3)); ax.plot(m.index.astype(str), m['mean'], marker='o'); ax.set_ylabel('churn rate')
plt.xticks(rotation=70); plt.title('Churn rate by snapshot month'); plt.show()

## 2. Leakage audit: `lifetime_months`
The dictionary says it is *total months the account was active, from a CRM export on 2026-01-05* — computed after the fact, so it can encode the outcome. Compare it with `tenure_months` (months so far, as of the snapshot).

In [ ]:
df['same'] = df.lifetime_months == df.tenure_months
print(df.groupby('same').churn_next_month.agg(['mean','size']))
print('AUC of the rule "lifetime == tenure": %.3f' % roc_auc_score(df.churn_next_month, df.same))

Perfect separation: every row whose lifetime equals its tenure churned, and no other row did. That is target leakage — at prediction time we cannot know the account's final lifetime. **`lifetime_months` is excluded from every model below.** Including it would give an AUC of 1.0 that is meaningless in production.

## 3. Features and validation scheme
Features known at snapshot time: plan, seats, fee, tenure, usage, tickets, plus a few derived from the account's *own earlier* snapshots (previous month's usage/tickets and the change in usage — backward-looking only). Calendar month is deliberately left out: the data has only two winters, so a model would just memorise the one winter spike.

Validation is a **time-based split**: train on snapshots up to 2025-05, test on 2025-06…2025-11. Accounts repeat across rows, so a random row split would let a model recognise accounts; the temporal split mimics real use. Note the test window contains no winter, so its base rate is lower than training's.

In [ ]:
df = df.sort_values(['account_ref','month']).reset_index(drop=True)
g = df.groupby('account_ref')
contig = g.month.diff().apply(lambda x: x.n if pd.notna(x) else np.nan) == 1   # previous row is exactly the prior month
df['usage_prev'] = g.usage_hours.shift().where(contig)
df['usage_delta'] = df.usage_hours - df.usage_prev
df['tickets_prev'] = g.support_tickets.shift().where(contig)
df['fee_per_seat'] = df.monthly_fee / df.seats
feats = ['plan','seats','monthly_fee','fee_per_seat','tenure_months','usage_hours','support_tickets','usage_prev','usage_delta','tickets_prev']
assert 'lifetime_months' not in feats
cut = pd.Period('2025-05','M')
tr, te = df[df.month <= cut], df[df.month > cut]
print(len(tr), len(te), '| train churn %.3f | test churn %.3f' % (tr.churn_next_month.mean(), te.churn_next_month.mean()))
print('accounts present in both train and test:', len(set(tr.account_ref) & set(te.account_ref)))

## 4. Univariate look (training period)

In [ ]:
for col in ['tenure_months','usage_hours','support_tickets','seats']:
    q = pd.qcut(tr[col], 5, duplicates='drop') if tr[col].nunique() > 6 else tr[col]
    print(tr.groupby(q, observed=True).churn_next_month.agg(['mean','size']).round(3).T, '\n')

## 5. Models vs. baselines

In [ ]:
X = lambda d: pd.get_dummies(d[feats], columns=['plan'], dtype=float)
Xtr, Xte = X(tr), X(te); ytr, yte = tr.churn_next_month, te.churn_next_month
models = {
 'logistic regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)),
 'gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
rows = [{'model':'baseline: prevalence', 'AUC':0.5, 'PR-AUC':yte.mean()}]
pred = {}
for n, mdl in models.items():
    mdl.fit(Xtr, ytr); pred[n] = mdl.predict_proba(Xte)[:,1]
    rows.append({'model':n, 'AUC':roc_auc_score(yte, pred[n]), 'PR-AUC':average_precision_score(yte, pred[n])})
pd.DataFrame(rows).round(3)

### Drift: the signal changes over time
A single raw feature (tickets) beats both full models on the test window. Per-feature AUC by quarter shows why: ticket count becomes more predictive over time while usage becomes less, and seats/fee/tenure are close to noise. The full models partly fit training-period patterns (including the winter spike) that do not carry over.

In [ ]:
q = df.month.dt.year.astype(str) + 'Q' + df.month.dt.quarter.astype(str)
print(pd.DataFrame({'churn': df.groupby(q).churn_next_month.mean(),
    'AUC tickets': df.groupby(q).apply(lambda d: roc_auc_score(d.churn_next_month, d.support_tickets)),
    'AUC -usage': df.groupby(q).apply(lambda d: roc_auc_score(d.churn_next_month, -d.usage_hours))}).round(3))
cols = ['support_tickets','usage_hours']
compact = LogisticRegression(max_iter=1000).fit(tr[cols], ytr)
name = 'compact logistic (tickets+usage)'
pred[name] = compact.predict_proba(te[cols])[:,1]
print(name, 'test AUC %.3f, PR-AUC %.3f' % (roc_auc_score(yte, pred[name]), average_precision_score(yte, pred[name])))

### Simple single-feature scores and a bootstrap CI (resampling accounts)

In [ ]:
for f, sign in [('usage_hours',-1), ('tenure_months',-1), ('support_tickets',1)]:
    print(f, 'AUC as a lone score: %.3f' % roc_auc_score(yte, sign*te[f]))
rng = np.random.default_rng(0); idx = te.reset_index(drop=True).groupby('account_ref').indices; accts = list(idx)
yv = yte.values
for n in pred:
    p = pred[n]; aucs = []
    for _ in range(300):
        s = np.concatenate([idx[a] for a in rng.choice(accts, len(accts))])
        if yv[s].min() != yv[s].max(): aucs.append(roc_auc_score(yv[s], p[s]))
    print('%s test AUC 95%% CI: %.3f-%.3f' % ((n,)+tuple(np.percentile(aucs,[2.5,97.5]))))

### Practical value: lift and calibration

In [ ]:
p = pred['gradient boosting']; o = np.argsort(-p); k = len(p)//10
print('Top-decile churn rate %.3f vs base %.3f -> lift %.2fx' % (yv[o[:k]].mean(), yv.mean(), yv[o[:k]].mean()/yv.mean()))
from sklearn.calibration import calibration_curve
a, b = calibration_curve(yv, p, n_bins=8, strategy='quantile')
print(pd.DataFrame({'predicted':b, 'observed':a}).round(3).T)

In [ ]:
from sklearn.inspection import permutation_importance
r = permutation_importance(models['gradient boosting'], Xte, yte, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(r.importances_mean, Xte.columns).sort_values().plot.barh(title='Permutation importance (test AUC drop)'); plt.show()

## Conclusions
- **Yes, moderately.** On a forward-in-time test (Jun–Nov 2025) models reach AUC of roughly 0.68–0.72 (test base rate 9%, PR-AUC about 0.2, top-decile lift about 2.8x). That is useful for prioritising outreach, not for confident individual predictions.
- **`lifetime_months` must not be used**: it perfectly reveals the label (post-hoc CRM export). Any near-perfect model using it is leakage.
- **Main drivers**: more support tickets and lower usage mean higher churn. Plan, seats, fee and tenure add little.
- **Caveats**: (1) the relationship drifts (tickets strengthen, usage weakens) and churn has a strong winter spike (Nov–Jan snapshots, up to ~50%) with only two winters observed, so winter performance is untested; (2) the compact tickets+usage model was chosen after seeing it beat the full models on this test window, so its score is somewhat optimistic — confirm on a later holdout; (3) predicted probabilities over-estimate churn on the test window because its base rate is lower, so recalibrate before treating them as absolute risks.